# 🫀 Heart Failure Risk Prediction & Comparative Model Benchmarking

This notebook demonstrates the end-to-end machine learning pipeline for predicting mortality risks in heart failure patients based on 12 clinical parameters.

## 📌 Project Requirements Checklist:
1. **Working Model Development**: Pretrained model & scalable inference pipeline.
2. **Deep Learning Implementations**: Artificial Neural Network (**ANN**), Long Short-Term Memory (**LSTM**), and Gated Recurrent Unit (**GRU**).
3. **Multiple Machine Learning Models**: Random Forest, XGBoost, Support Vector Machine (SVM), Decision Tree, K-Nearest Neighbors (KNN), and Logistic Regression.
4. **Comparative Performance Analysis**: Metrics comparison across Accuracy, Precision, Recall, F1-Score, ROC-AUC, and execution time.
5. **Identification & Justification of Best Model**: Clinical domain rationale and metric evaluation.

--- 
## 1. Environment Setup & Data Loading

In [ ]:
import os
import sys
import site
sys.path.append(site.getusersitepackages())

import time
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix
)

# ML Models
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

# Deep Learning PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Set Random Seeds
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
print("✅ Libraries imported successfully.")

In [ ]:
df = pd.read_csv("heart_failure.csv")
print(f"Dataset Shape: {df.shape}")
display(df.head())
print("\nTarget Class Distribution:")
print(df["DEATH_EVENT"].value_counts(normalize=True))

--- 
## 2. Preprocessing & Feature Scaling

In [ ]:
X = df.drop("DEATH_EVENT", axis=1)
y = df["DEATH_EVENT"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

joblib.dump(scaler, "scaler.pkl")
print(f"Train samples: {len(X_train)}, Test samples: {len(X_test)}")

--- 
## 3. Machine Learning Models Training & Benchmarking

In [ ]:
results = {}
prob_predictions = {}
cm_dict = {}
trained_objects = {}

def evaluate_model(name, y_true, y_pred, y_prob, train_time):
    from sklearn.metrics import mean_absolute_error, mean_squared_error
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    auc = roc_auc_score(y_true, y_prob)
    mae = mean_absolute_error(y_true, y_prob)
    mse = mean_squared_error(y_true, y_prob)
    rmse = np.sqrt(mse)
    cm = confusion_matrix(y_true, y_pred)
    
    results[name] = {
        "Accuracy": round(acc * 100, 2),
        "Precision": round(prec * 100, 2),
        "Recall": round(rec * 100, 2),
        "F1-score": round(f1 * 100, 2),
        "MAE": round(mae, 4),
        "MSE": round(mse, 4),
        "RMSE": round(rmse, 4),
        "Training Time": round(train_time, 4)
    }
    prob_predictions[name] = y_prob
    cm_dict[name] = cm
    print(f"[{name}] Acc: {acc*100:.2f}% | F1: {f1*100:.2f}% | MAE: {mae:.4f} | MSE: {mse:.4f} | RMSE: {rmse:.4f}")

ml_models = {
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=5, random_state=SEED),
    "XGBoost": XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=4, random_state=SEED, eval_metric='logloss'),
    "SVM (RBF)": SVC(probability=True, kernel='rbf', C=1.0, random_state=SEED),
    "Decision Tree": DecisionTreeClassifier(max_depth=4, random_state=SEED),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Logistic Regression": LogisticRegression(random_state=SEED)
}

for name, model in ml_models.items():
    t0 = time.time()
    model.fit(X_train_scaled, y_train)
    dt = time.time() - t0
    
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    evaluate_model(name, y_test, y_pred, y_prob, dt)
    trained_objects[name] = model

--- 
## 4. Deep Learning Models (ANN, LSTM, GRU)

In [ ]:
X_tr_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_tr_t = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)
X_te_t = torch.tensor(X_test_scaled, dtype=torch.float32)
y_te_t = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)

train_dataset = TensorDataset(X_tr_t, y_tr_t)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

class ANNModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 32), nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.net(x)

class RecurrentModel(nn.Module):
    def __init__(self, cell_type="LSTM", input_dim=1, hidden_dim=64):
        super().__init__()
        if cell_type == "LSTM":
            self.rnn = nn.LSTM(input_size=input_dim, hidden_size=hidden_dim, num_layers=2, batch_first=True, dropout=0.2)
        else:
            self.rnn = nn.GRU(input_size=input_dim, hidden_size=hidden_dim, num_layers=2, batch_first=True, dropout=0.2)
        self.fc = nn.Sequential(nn.Linear(hidden_dim, 32), nn.ReLU(), nn.Linear(32, 1))
    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(out[:, -1, :])

dl_histories = {}
def train_dl(name, model, is_recurrent=False, epochs=100, lr=0.005):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    l_hist, a_hist = [], []
    t0 = time.time()
    for ep in range(epochs):
        model.train()
        r_loss, correct, total = 0.0, 0, 0
        for bx, by in train_loader:
            if is_recurrent: bx = bx.unsqueeze(-1)
            optimizer.zero_grad()
            out = model(bx)
            loss = criterion(out, by)
            loss.backward()
            optimizer.step()
            r_loss += loss.item() * bx.size(0)
            preds = (torch.sigmoid(out) >= 0.5).float()
            correct += (preds == by).sum().item()
            total += by.size(0)
        l_hist.append(r_loss/total)
        a_hist.append(correct/total)
    dt = time.time() - t0
    model.eval()
    with torch.no_grad():
        tx = X_te_t.unsqueeze(-1) if is_recurrent else X_te_t
        probs = torch.sigmoid(model(tx)).numpy().flatten()
        preds = (probs >= 0.5).astype(int)
    evaluate_model(name, y_test, preds, probs, dt)
    dl_histories[name] = {"loss": l_hist, "accuracy": a_hist}

train_dl("ANN", ANNModel(12), is_recurrent=False)
train_dl("LSTM", RecurrentModel("LSTM"), is_recurrent=True)
train_dl("GRU", RecurrentModel("GRU"), is_recurrent=True)

--- 
## 5. Comprehensive Comparative Analysis & Visualizations

In [ ]:
df_results = pd.DataFrame(results).T.sort_values(by="F1-Score", ascending=False)
print("=== COMPARATIVE PERFORMANCE TABLE ===")
display(df_results)

In [ ]:
# Save Best Model
best_model_name = df_results.index[0]
print(f"🏆 Best Model: {best_model_name}")
if best_model_name in trained_objects:
    joblib.dump(trained_objects[best_model_name], "heart_model.pkl")
    print(f"Saved {best_model_name} to heart_model.pkl")